# EthicAgent — Scenario Analysis

This notebook demonstrates how to run and analyze the built-in ethical scenarios across all four domains:
1. **Healthcare Triage** — Medical resource allocation and patient care
2. **Loan Approval** — Financial lending and fair credit decisions
3. **Hiring Decision** — Employment fairness and anti-discrimination
4. **Disaster Response** — Emergency resource distribution and triage

---

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from collections import Counter

from ethicagent.scenarios import (
    HealthcareTriageScenario,
    LoanApprovalScenario,
    HiringDecisionScenario,
    DisasterResponseScenario,
)
from ethicagent.scenarios.base_scenario import ScenarioCase
from ethicagent.agents.ethical_reasoner import EthicalReasonerAgent

print('Scenario modules loaded successfully!')

## 1. Loading Built-in Scenarios

Each domain has 30+ pre-defined test cases with expected verdicts and EDS ranges.

In [ ]:
# Initialize all scenarios
scenarios = {
    'Healthcare': HealthcareTriageScenario(),
    'Finance': LoanApprovalScenario(),
    'Hiring': HiringDecisionScenario(),
    'Disaster': DisasterResponseScenario(),
}

# Summary table
print(f'{"Domain":12s} | {"Cases":>6s} | {"Domain Tag":>12s}')
print('-' * 38)
total = 0
for name, scenario in scenarios.items():
    cases = scenario.get_test_cases()
    total += len(cases)
    print(f'{name:12s} | {len(cases):6d} | {scenario.domain:>12s}')
print('-' * 38)
print(f'{"TOTAL":12s} | {total:6d} |')

## 2. Exploring a Domain — Healthcare Triage

In [ ]:
healthcare = scenarios['Healthcare']
cases = healthcare.get_test_cases()

# Show first 5 cases
print(f'Healthcare Triage — First 5 Cases\n')
for case in cases[:5]:
    print(f'ID: {case.case_id}')
    print(f'  Task: {case.task}')
    print(f'  Expected: {case.expected_verdict}')
    print(f'  Context: {case.context}')
    print()

In [ ]:
# Verdict distribution for healthcare
verdicts = Counter(c.expected_verdict for c in cases)
print('Healthcare Verdict Distribution:')
for verdict, count in sorted(verdicts.items()):
    bar = '█' * count
    print(f'  {verdict:12s}: {count:3d} {bar}')

## 3. Cross-Domain Verdict Distribution

In [ ]:
# Compare verdict distributions across all domains
print(f'{"Domain":12s} | {"approve":>8s} | {"escalate":>8s} | {"reject":>8s} | {"hard_block":>10s}')
print('-' * 60)

for name, scenario in scenarios.items():
    cases = scenario.get_test_cases()
    dist = Counter(c.expected_verdict for c in cases)
    print(
        f'{name:12s} | '
        f'{dist.get("approve", 0):8d} | '
        f'{dist.get("escalate", 0):8d} | '
        f'{dist.get("reject", 0):8d} | '
        f'{dist.get("hard_block", 0):10d}'
    )

## 4. Running Scenario Evaluation

Evaluate each scenario case using the EDS formula and compare with expected verdicts.

In [ ]:
reasoner = EthicalReasonerAgent()

# Domain weight configs
domain_weight_map = {
    'healthcare': {'deontological': 0.35, 'consequentialist': 0.25, 'virtue_ethics': 0.20, 'contextual': 0.20},
    'finance':    {'deontological': 0.20, 'consequentialist': 0.25, 'virtue_ethics': 0.35, 'contextual': 0.20},
    'hiring':     {'deontological': 0.15, 'consequentialist': 0.20, 'virtue_ethics': 0.40, 'contextual': 0.25},
    'disaster':   {'deontological': 0.20, 'consequentialist': 0.35, 'virtue_ethics': 0.15, 'contextual': 0.30},
}

def evaluate_scenario(scenario, domain_name):
    """Evaluate all cases in a scenario and return results."""
    cases = scenario.get_test_cases()
    weights = domain_weight_map.get(scenario.domain, domain_weight_map['healthcare'])
    results = []
    
    for case in cases:
        # Use pre-defined scores from expected_eds or generate from context
        scores = case.context.get('philosophy_scores', {
            'deontological': case.context.get('deontological', 0.7),
            'consequentialist': case.context.get('consequentialist', 0.7),
            'virtue_ethics': case.context.get('virtue_ethics', 0.7),
            'contextual': case.context.get('contextual', 0.7),
        })
        
        eds = reasoner.compute_eds(scores, weights)
        verdict = reasoner.determine_verdict(scores, weights)
        match = verdict == case.expected_verdict
        
        results.append({
            'case_id': case.case_id,
            'task': case.task[:50],
            'expected': case.expected_verdict,
            'predicted': verdict,
            'eds': eds,
            'match': match,
        })
    
    return results

# Run evaluation for healthcare
healthcare_results = evaluate_scenario(scenarios['Healthcare'], 'healthcare')
accuracy = sum(1 for r in healthcare_results if r['match']) / len(healthcare_results)
print(f'Healthcare Evaluation: {accuracy:.1%} accuracy ({sum(1 for r in healthcare_results if r["match"])}/{len(healthcare_results)})')
print(f'\nSample results:')
for r in healthcare_results[:8]:
    status = '✓' if r['match'] else '✗'
    print(f'  {status} {r["case_id"]:20s} EDS={r["eds"]:.3f} expected={r["expected"]:12s} got={r["predicted"]}')

## 5. Loading JSON Test Data

EthicAgent includes JSON data files with 50+ test cases per domain.

In [ ]:
# Load JSON scenario data
data_dir = Path('..') / 'data' / 'scenarios'
json_files = sorted(data_dir.glob('*.json'))

print(f'Found {len(json_files)} JSON data files:\n')
for f in json_files:
    with open(f) as fh:
        data = json.load(fh)
    cases = data.get('cases', data.get('scenarios', []))
    print(f'  {f.name:30s}: {len(cases):3d} cases')
    
    # Show sample case
    if cases:
        sample = cases[0]
        print(f'    Sample: {sample.get("task", sample.get("description", "N/A"))[:60]}')
        print()

## 6. EDS Score Distribution Analysis

In [ ]:
# Analyze EDS distribution for all scenarios that have expected_eds
for name, scenario in scenarios.items():
    cases = scenario.get_test_cases()
    eds_values = [c.expected_eds for c in cases if c.expected_eds is not None]
    
    if eds_values:
        avg_eds = sum(eds_values) / len(eds_values)
        min_eds = min(eds_values)
        max_eds = max(eds_values)
        print(f'{name}: avg={avg_eds:.3f}, min={min_eds:.3f}, max={max_eds:.3f}, n={len(eds_values)}')
    else:
        print(f'{name}: No expected EDS values available')

## 7. Scenario Categories Deep Dive

In [ ]:
# Analyze cases by category within each domain
for name, scenario in scenarios.items():
    cases = scenario.get_test_cases()
    categories = Counter(c.context.get('category', 'uncategorized') for c in cases)
    
    print(f'\n{name} Categories:')
    for cat, count in categories.most_common():
        print(f'  {cat:30s}: {count:3d}')

## 8. Running Full Domain Comparison

In [ ]:
# Full evaluation across all domains
all_results = {}
for name, scenario in scenarios.items():
    results = evaluate_scenario(scenario, name.lower())
    all_results[name] = results
    accuracy = sum(1 for r in results if r['match']) / len(results) if results else 0
    print(f'{name:12s}: {accuracy:.1%} accuracy, {len(results)} cases')

# Overall accuracy
total_match = sum(1 for domain_results in all_results.values() for r in domain_results if r['match'])
total_cases = sum(len(domain_results) for domain_results in all_results.values())
print(f'\nOverall: {total_match/total_cases:.1%} ({total_match}/{total_cases})')

## 9. Mismatch Analysis

Examine cases where the EDS verdict disagrees with the expected verdict.

In [ ]:
# Show mismatches
for name, results in all_results.items():
    mismatches = [r for r in results if not r['match']]
    if mismatches:
        print(f'\n{name} Mismatches ({len(mismatches)}):')
        for r in mismatches[:5]:
            print(f'  {r["case_id"]}: expected={r["expected"]:12s} got={r["predicted"]:12s} EDS={r["eds"]:.3f}')

---

**Next**: [03_evaluation_results.ipynb](03_evaluation_results.ipynb) — Full benchmark results with statistical analysis